# HyperPE Corner Plot: Original vs Combined Posteriors (10 events)

Compare EOSFIT and spectral HyperPE runs with original (~5k samples/event) vs combined (~10k samples/event) posterior samples. Parameters match the `plot_parameters` in `HyperPE_eosfit.py` and `HyperPE_spectral_SEOBNR.py`.

In [ ]:
import os, csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import bilby
from wcosmo.astropy import Planck18

RUNDIR = '/scratch/gpfs/ANDREASB/fy6204/GW/Workspace'
os.chdir(RUNDIR)
print("Imports OK")

In [ ]:
def load_posterior_df(path):
    return pd.read_csv(path)

def make_lightweight_result(df, params, label):
    priors = {}
    for p in params:
        x = df[p].dropna().to_numpy(dtype=float)
        lo, hi = np.percentile(x, [0.1, 99.9]) if len(x) > 0 else (0, 1)
        priors[p] = bilby.core.prior.Uniform(lo, hi, name=p)
    return bilby.core.result.Result(
        label=label, outdir='.', sampler='dynesty',
        search_parameter_keys=params, priors=priors,
        posterior=df[params].copy(),
        parameter_labels=[p for p in params],
        parameter_labels_with_unit=[p for p in params],
    )

def add_truth_lines(fig, plot_params, truths):
    ndim = len(plot_params)
    axes = np.array(fig.axes).reshape((ndim, ndim))
    for i, yp in enumerate(plot_params):
        yt = truths.get(yp)
        if yt is not None and np.isfinite(yt):
            axes[i, i].axvline(yt, color='red', lw=1.5)
        for j in range(i):
            xp = plot_params[j]
            xt = truths.get(xp)
            if xt is not None and np.isfinite(xt):
                axes[i, j].axvline(xt, color='red', lw=1.0)
            if yt is not None and np.isfinite(yt):
                axes[i, j].axhline(yt, color='red', lw=1.0)
    return fig

def make_overlay_corner(path_orig, path_comb, params, truths, save_path):
    df_orig = load_posterior_df(path_orig)
    df_comb = load_posterior_df(path_comb)
    r1 = make_lightweight_result(df_orig, params, 'Original (5k/ev)')
    r2 = make_lightweight_result(df_comb, params, 'Combined (10k/ev)')
    fig = bilby.core.result.plot_multiple(
        [r1, r2], labels=['Original (5k/ev)', 'Combined (10k/ev)'],
        parameters=params, save=False, bins=30, plot_datapoints=False,
        smooth=0.9, quantiles=[0.16, 0.84], colors=['C0', 'C3'],
    )
    add_truth_lines(fig, params, truths)
    fig.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Saved {save_path}")
    return fig

## EOSFIT (11 parameters)

Matching `HyperPE_eosfit.py` line 940: `lamb, mu_m, sigma_m, mu_H, sigma_H, mu_a0, sigma_a0, mu_a1, sigma_a1, mu_a2, sigma_a2`

In [ ]:
EOSFIT_PARAMS = [
    "lamb", "mu_m", "sigma_m", "mu_H", "sigma_H",
    "mu_a0", "sigma_a0", "mu_a1", "sigma_a1", "mu_a2", "sigma_a2",
]
EOSFIT_TRUTHS = {
    "lamb": 0.0, "mu_m": 1.33, "sigma_m": 0.09,
    "mu_H": float(Planck18.H0.value), "sigma_H": 0.0,
    "mu_a0": 0.0, "sigma_a0": 0.0,
    "mu_a1": 0.0, "sigma_a1": 0.0,
    "mu_a2": 0.0, "sigma_a2": 0.0,
}

make_overlay_corner(
    path_orig='outputs/outdir_hyperpe_run_SEOBNR_10/hyperpe_SEOBNR_10_full_posterior_extra_statistics.csv',
    path_comb='outputs/outdir_hyperpe_SEOBNR_10_combined/hyperpe_SEOBNR_10_combined_full_posterior_extra_statistics.csv',
    params=EOSFIT_PARAMS, truths=EOSFIT_TRUTHS,
    save_path='fig_corner_eosfit_10ev_original_vs_combined.png',
)

## Spectral (4 parameters)

Matching `HyperPE_spectral_SEOBNR.py` line 646: `H0, mu_m, sigma_m, lamb`

In [ ]:
SPECTRAL_PARAMS = ["H0", "mu_m", "sigma_m", "lamb"]
SPECTRAL_TRUTHS = {"H0": float(Planck18.H0.value), "mu_m": 1.33, "sigma_m": 0.09, "lamb": 0.0}

make_overlay_corner(
    path_orig='outputs/outdir_spectral_siren_seobnr_10/spectral_siren_seobnr_10_full_posterior_extra_statistics.csv',
    path_comb='outputs/outdir_spectral_siren_seobnr_10_combined/spectral_siren_seobnr_10_combined_full_posterior_extra_statistics.csv',
    params=SPECTRAL_PARAMS, truths=SPECTRAL_TRUTHS,
    save_path='fig_corner_spectral_10ev_original_vs_combined.png',
)

## Note

The original and combined posteriors are nearly identical. Biases come from the single-event PE likelihood, not from insufficient posterior samples in the HyperPE Monte Carlo integration.